# Tech Note - FFT

FFTを使うための必要事項をこの技術ノートに記す。<br>
特に、周波数分解能について、参考書やWeb記事を引用しながら整理する。

# 1. FFTの周波数分解能は、複素フーリエ級数の時間窓長で決まる

## 1.1 複素指数関数（オイラーの公式）
波形を数学的に扱いやすくするために、一般に複素指数関数が使われる。実数 $\theta$ と ネイピア数 $e$ = 2.71828 を用いて：

$$
e^{i\theta} = \cos \theta + i \sin \theta \quad \quad (1.1)
$$

## 1.2 複素フーリエ級数
複素指数関数 $e^{i\theta}$ を使うと、時間窓 $0 \le t < T$ で "連続的な" 信号 $x(t)$ は複素フーリエ級数で次のように表される：

$$
\begin{align}
x(t) &= \cdots + c_{-2} e^{i \frac{2 \pi}{T} \times (-2t)} + c_{-1} e^{i \frac{2 \pi}{T} \times (-t)} \notag \\
&+ c_0 \times 1 \notag \\
&+ c_{1} e^{i \frac{2 \pi}{T} \times t} + c_{2} e^{i \frac{2 \pi}{T} \times 2t} + \cdots \notag \quad \quad (1.2)
\end{align}
$$

（参考：道具としてのフーリエ解析、涌井良幸・涌井貞美、日本実業出版社、2014年）

各振動数 $\omega$ [rad/s] と 周波数 $f$ [Hz] の関係 $\omega = 2 \pi f$ を念頭に置いて、式(1.2)の指数部に注目する。

$ \cdots $ <br>
$ i \frac{2 \pi}{T} \times (-2t) $ <br>
$ i \frac{2 \pi}{T} \times (-t) $ <br>
$ i \frac{2 \pi}{T} \times 0t $ <br>
$ i \frac{2 \pi}{T} \times t $ <br>
$ i \frac{2 \pi}{T} \times 2t $ <br>
$ \cdots $

複素フーリエ級数は、時間窓長 $T$ の逆数 $1/T$ を基準として、その整数倍の周波数成分の和で構成されている。

## 1.3 複素フーリエ級数の周波数分解能

これより、時間窓 $0 \le t < T$ で "連続的な" 信号 $x(t)$ 複素フーリエ級数で表す場合、周波数分解能 $\Delta f$ は時間窓長 $T$ によって決まることが分かる。
$$
\Delta f = \frac{1}{T} \quad \quad (1.3)
$$

**注意#1** <br>
これはDFTやFFTを適用する離散信号に対するものではなく、連続波形を複素フーリエ展開するときの理論値である。<br>
離散信号のサンプリング周波数（後述）と混同しないよう、はっきりと区別すること。

**注意#2** <br>
周波数は "数値が小さいほど"、"ゆっくりとした（変化ペースが遅い）" 波形を表す。<br>
"周波数分解能"とは、10.0Hzと10.1Hzのような「非常に近い周波数の"小さな違い"」を聞き分ける、あるいは、表現し分ける能力である。

* 正：周波数の分解能が"高い" --> 周波数の"小さな差異成分"（＝周波数が"低い"、変化ペースが遅くて区別しにくい成分）を識別・表現できる
* 誤：周波数の分解能が"高い" --> 周波数が"高い"波（＝変化ペースが"速く"認識しにくい）を識別・表現できる<br>
　　　　　　　　　　　　　　　（※これは、周波数分解能ではなくサンプリング周波数が高い状態（後述））

（イメージ：周波数グラフ（スペクトル）の画素の細かさ）
* 周波数分解能が高い --> 「周波数軸の目盛りが細かい定規」「4Kディスプレイ (文字や映像の細部までくっきりと表示)」<br>
　　　　　　　　　　　10.0Hzと10.1Hzのような「非常に近い周波数の違い」を区別。
* 周波数分解能が低い --> 「周波数軸の目盛りが粗い定規」「モザイク画」<br>
　　　　　　　　　　　10.0Hzと10.5Hzが「約10Hzの成分」としてひとまとめにされてしまう。隣り合う周波数がぼやけてくっついて見える。

**注意3** <br>
時間窓長 $T$ が長いほど周波数分解能が高くなることについて、直観的な理解：<br>
「ゆっくりとした変化に気づくには、時間をかけてじっくりと観察する必要がある」

1. 低い周波数（ゆっくりな波）そのものを見るため
    * 低い周波数（例えば 0.1 Hz）の波は、1回の振動に 10秒 かかる
    * もし観察時間（時間窓 $T$）が 1秒しかなければ、その波のほんの一部（坂道のような部分）しか見えない
    * それが「波」なのか「ただの右肩上がりの直線」なのか区別がつかない
    * 「ゆっくりした波」を「波」として認識するには、少なくとも1周期分くらいは「じっくり観察（長い $T$）」する必要がある

2. 周波数の「違い（ズレ）」を見るため（うなり）
    * これが周波数分解能の核心。例えば「10.0 Hz」と「10.1 Hz」という 「微差」 を見分けるケースを想像する
    * この2つの波は、短時間（0.1秒など）だけ見ても、ほとんど重なっていて区別がつかない
    * しかし、10秒間じっくり観察し続けると、10.1 Hzの方が少しずつ先に進んで、1周分の「差」がつく（位相のズレ）
    * この 「わずかなズレが蓄積して、違いが明確になるまで待つ」 ということもまた、「時間をかけてじっくり観察する」ことの本質

要するに「じっくり待つことで違いが顕在化する」ということ。

In [ ]:
# 例：時間窓長T=1秒のとき
T = 1
f_delta = 1 / T
print(f"時間窓長 T={T} 秒")
print(f"周波数分解能 {f_delta=} Hz")
print(f"x2 周波数 {2*f_delta=} Hz")
print(f"x3 周波数 {3*f_delta=} Hz")

In [ ]:
# 例：時間窓長T=10秒のとき
T = 10
f_delta = 1 / T
print(f"時間窓長 T={T} 秒")
print(f"周波数分解能 {f_delta=} Hz")
print(f"x2 周波数 {2*f_delta=} Hz")
print(f"x3 周波数 {3*f_delta=} Hz")

# 2. 離散信号と周波数分解能

## 2.1 離散信号：時間窓長とサンプリング点数との関係

コンピュータで処理する信号は離散値である。離散信号における時間窓長$T$を確認する。

サンプリング点数 $N$・・・FFTの時間窓での有限点数

FFTの時間窓長 $T$ は、サンプリング点数 $N$ とサンプリング周波数 $f_s$（サンプリング間隔 $\Delta t$）で決まる。

* $N$：サンプリング点数
* $\Delta t$ [s] ：サンプリング間隔
* $f_s$ [Hz]：サンプリング周波数
* $T$ [s]：時間窓長

$$
T = N \Delta t = \frac{N}{f_s} \quad \quad (2.1)
$$

例：
* $N$ = 2048
* $f_s$ = 51.2 [kHz]

$$
T = \frac{2048}{51.2 \times 1000} = 0.04 [\mathrm{s}] = 40 [\mathrm{ms}]
$$

引用：小野測器 時間窓長とサンプリング点数との関係は？<br>
https://www.onosokki.co.jp/HP-WK/c_support/faq/fft_common/fft_analys_3.htm

In [ ]:
2048 / (51.2 * 1000)

## 2.2 離散信号：周波数分解能

複素フーリエ級数の周波数分解能の式 (式1.3)に、離散信号の式 (式2.1) を代入して、離散信号における周波数分解能を得る：

* $\Delta f$ [Hz]：周波数分解能

$$
\Delta f = \frac{1}{T} = \frac{1}{N \Delta t} = \frac{f_s}{N} \quad \quad (2.2)
$$

周波数分解能を高める（$\Delta f$ を小さくする）には、時間窓長 $T$ を大きくする必要がある。そのためには、
* サンプリング周波数 $f_s$ を小さくする
* サンプリング点数 $N$ を大きくする

通常は、分析周波数レンジを決めると必然的にサンプリング周波数が自動的に決定される。<br>
そのため現実的には、周波数分解能は主としてサンプリング点数 $N$ に依存する。

引用：小野測器 周波数分解能はどのように決めるのか？<br>
https://www.onosokki.co.jp/HP-WK/c_support/faq/fft_common/fft_analys_4.htm

## 2.3 離散信号：高周波信号への対応能力

ナイキスト周波数 $f_n$ は、サンプリング周波数の1/2の周波数：
$$
f_n = \frac{f_s}{2} \quad \quad (2.3)
$$

ナイキスト周波数 $f_n$ より高い周波数の信号をサンプリングした場合、折り返し（エイリアシング）が生じ、再生時に、元信号を忠実に再現できない。

高周波信号を正しく分析するには、サンプリング周波数を高める必要がある。<br>

（イメージ：ストロボ効果／ワゴンホイール効果）
* サンプリング周波数が高い --> 元信号の"完全再現"
* サンプリング周波数が不足 --> 再生時に"エイリアシング（折り返し）成分が混入"

引用：Wikipedia ナイキスト周波数<br>
https://ja.wikipedia.org/wiki/%E3%83%8A%E3%82%A4%E3%82%AD%E3%82%B9%E3%83%88%E5%91%A8%E6%B3%A2%E6%95%B0

引用：Wikipedia ストロボ効果<br>
https://ja.wikipedia.org/wiki/%E3%82%B9%E3%83%88%E3%83%AD%E3%83%9C%E5%8A%B9%E6%9E%9C

In [ ]:
# 例：サンプリング期間=10秒、サンプリング周波数=1Hz
fs = 1  # Hz
ds = 1/fs  # 秒
f_nyquest = fs / 2  # ナイキスト周波数
print(f"サンプリング周波数 fs={fs}Hz, サンプリング間隔 ds={ds}秒, ナイキスト周波数 f_nyquest={f_nyquest}Hz")
N = 10  # 点
print(f"10秒間サンプリング数 N={N}点")
T = N / fs  # 秒
df = 1 / T  # Hz
print(f"時間窓長 T={T}秒, 周波数分解能 df={df}Hz")

In [ ]:
# 例：サンプリング期間=10秒、サンプリング周波数=10Hz
fs = 10  # Hz
ds = 1/fs  # 秒
f_nyquest = fs / 2  # ナイキスト周波数
print(f"サンプリング周波数 fs={fs}Hz, サンプリング間隔 ds={ds}秒, ナイキスト周波数 f_nyquest={f_nyquest}Hz")
N = 100  # 点
print(f"10秒間サンプリング数 N={N}点")
T = N / fs  # 秒
df = 1 / T  # Hz
print(f"時間窓長 T={T}秒, 周波数分解能 df={df}Hz")

# 3. 周波数について、明確に区別すべきこと

「周波数分解能」と「高周波への対応能力」は、混同しがちだが、明確に区別する必要がある。

## 3.1 周波数分解能（複素フーリエ級数の理論値）
時間窓長が長い --> 周波数分解能が高い （「周波数の画素密度（解像度）」。隣り合う音程を聞き分ける「耳の良さ」）

## 3.2 高周波への対応能力（サンプリングの細かさ）
高速サンプリング --> 高周波成分を識別可能（「動体視力」。速い動きを正確に捉える力（低いと、速い動きが変な動きに化けて見える））

# 4. 実習：NumPyのFFT

## 4.1 サンプル時系列データ

### 4.1.1 個別正弦波

In [ ]:
import numpy as np

# 時系列
dt = 0.01  # サンプリング間隔 [s]
fs = 1 / dt  # サンプリング周波数 [Hz]
T = 10.24   # 全体時間長 [s] (データ数を2のべき乗にする (2^10=1024))
time_series = np.arange(0, T, dt)
print(f"{time_series.shape[-1]=}")

# 正弦波#1
a1 = 0.1  # 振幅 [m]
f1 = 1  # 周波数 [Hz]
sin_wave1 = a1 * np.sin(2 * np.pi * f1 * time_series)

# 正弦波#2
a2 = 0.075  # 振幅 [m]
f2 = 3  # 周波数 [Hz]
sin_wave2 = a2 * np.sin(2 * np.pi * f2 * time_series)

# 正弦波#3
a3 = 0.05  # 振幅 [m]
f3 = 5  # 周波数 [Hz]
sin_wave3 = a3 * np.sin(2 * np.pi * f3 * time_series)

# プロット
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=time_series, y=sin_wave1, name='正弦波1Hz'))
fig.add_trace(go.Scatter(x=time_series, y=sin_wave2, name='正弦波3Hz'))
fig.add_trace(go.Scatter(x=time_series, y=sin_wave3, name='正弦波5Hz'))
fig.update_layout(showlegend=True, title='正弦波', xaxis_title='時間 [s]', yaxis_title='振幅 [m]')
fig.show()

### 4.1.2 合成正弦波

In [ ]:
# 正弦波#4 = #1 + #2 + #3
sin_wave4 = sin_wave1 + sin_wave2 + sin_wave3

# プロット
fig = go.Figure()
fig.add_trace(go.Scatter(x=time_series, y=sin_wave4, name='合成正弦波'))
fig.update_layout(showlegend=True, title='合成正弦波', xaxis_title='時間 [s]', yaxis_title='振幅 [m]')
fig.show()

## 4.2 NumPyのFFT

NumPy: Discrete Fourier Transform<br>
https://numpy.org/doc/stable/reference/routines.fft.html

※numpy.fftはscipy.fftの基本セット。包括的なFFT機能はSciPyにある。<br>
"The SciPy module scipy.fft is a more comprehensive superset of numpy.fft, which includes only a basic set of routines."

numpy.fft.fft<br>
https://numpy.org/doc/stable/reference/generated/numpy.fft.fft.html#numpy.fft.fft

numpy.fft.fftfreq<br>
https://numpy.org/doc/stable/reference/generated/numpy.fft.fftfreq.html#numpy.fft.fftfreq

確認結果：<br>
* 周波数スペクトルには、サンプル時系列データの周波数1Hz、3Hz、5Hzがピークとして表れている。
* NumPyを使って、正しくFFT処理を実行できている。

In [ ]:
import numpy as np

# Fast Fourier Transform (FFT) 
# spectrum : complex ndarray (a: array_like)
sp = np.fft.fft(sin_wave4)
print(f"{type(sp)=}")
print(f"{type(sp[0])=}")
print(f"{sp.shape=}")

# sample frequencies (n: window length, d: sample spacing)
# f = [0, 1, ...,   n/2-1,     -n/2, ..., -1] / (d*n)   if n is even
# f = [0, 1, ..., (n-1)/2, -(n-1)/2, ..., -1] / (d*n)   if n is odd
freq = np.fft.fftfreq(n=time_series.shape[-1], d=dt)

T = time_series.shape[-1] * dt # [s]
df = 1 / T  # [Hz]
print(f"時間窓長 T={T}秒, 周波数分解能 df={df}Hz")
fs = 1 / dt  # [Hz]
f_nyquest = fs / 2  # ナイキスト周波数
print(f"サンプリング周波数 fs={fs}Hz, ナイキスト周波数 f_nyquest={f_nyquest}Hz")

import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=freq, y=np.abs(sp), name='周波数スペクトル'))
fig.update_layout(showlegend=True, title='周波数スペクトル (numpy.fft)', xaxis_title='周波数 [Hz]', yaxis_title='振幅')
fig.show()

## 4.3 NumPyのInverse-FFT

numpy.fft.ifft<br>
https://numpy.org/doc/stable/reference/generated/numpy.fft.ifft.html#numpy.fft.ifft

確認結果：<br>
* 元の合成正弦波が再現されている。
* NumPyを使って、正しくInverse-FFT処理を実行できている。

In [ ]:
s = np.fft.ifft(sp)
print(f"{type(s)=}")
print(f"{type(s[0])=}")

import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=time_series, y=s.real, name='逆フーリエ変換結果 (real)'))
fig.add_trace(go.Scatter(x=time_series, y=s.imag, name='逆フーリエ変換結果 (imag)', mode='lines', line=dict(dash='dash')))
fig.update_layout(showlegend=True, title='逆フーリエ変換結果 (numpy.ifft)', xaxis_title='時間 [s]', yaxis_title='振幅')
fig.show()

In [ ]:
# 複素数の例
c = 1+1j
print(f"{type(c)=}")
print(f"{c=}")
print(f"{c.real=}")
print(f"{c.imag=}")
print(f"{abs(c)=}")

# 5. 実習：SciPyのFFT

SciPy と NumPy の違い

基本的な使い方は同じだが、SciPyの scipy.fft は、内部的に最適化されており、特にデータ点数が「2の累乗」でない場合でも高速に計算してくれる機能（バックエンドの工夫）が含まれている。実務で大規模なデータを扱う際は scipy を使うのが無難。

## 5.1 サンプル時系列データ

NumPyのFFTと同じデータ (4.1) を使う。

## 5.2 SciPyのFFT

scipy.fft<br>
https://docs.scipy.org/doc/scipy/tutorial/fft.html

FFT:
$$
y[k] = \sum_{n=0}^{N-1} e^{-2\pi j \frac{kn}{N}} x[n]
$$

※FFT (DFT)の式は総和 $\sum$ を取るため、データ点数$N$が増えると、計算結果の数値も$N$倍になる。

※正規化の注意事項<br>
片側正規化用にすべての成分に対して一律に 2/N を掛けて正規化する場合、実は 直流成分（0Hz）だけは例外。
* 交流成分（AC）： 正と負の周波数に半分ずつパワーが分かれるため、片側にする際に2倍する（2/N）のが正解。
* 直流成分（DC）： 0Hzは分割されずにそのまま存在するため、2倍する必要はなく、単純平均（1/N）が正解。

Inverse-FFT:
$$
x[n] = \frac{1}{N} \sum_{k=0}^{N-1} e^{2 \pi j \frac{kn}{N}} y[k]
$$

scipy.fft.fft<br>
https://docs.scipy.org/doc/scipy/reference/generated/scipy.fft.fft.html#scipy.fft.fft

scipy.fft.fftfreq<br>
https://docs.scipy.org/doc/scipy/reference/generated/scipy.fft.fftfreq.html#scipy.fft.fftfreq

確認結果：<br>
* 周波数スペクトル：
    * ピーク周波数≒サンプル時系列データの周波数1、3、5 [Hz]
    * 正規化（片側）振幅≒サンプル時系列データの振幅0.1、0.075、0.5 [m]
* SciPyを使って、正しくFFT処理を実行できている。

In [ ]:
from scipy.fft import fft, fftfreq
import numpy as np

# Number of sample points
N = time_series.shape[-1]
print(f"{N=}")

# sample spacing
d = dt  # [s]

# spectrum : complex ndarray (a: array_like)
yf = fft(sin_wave4)
print(f"{type(yf)=}")
print(f"{type(yf[0])=}")
print(f"{yf.shape=}")

# 振幅正規化（両側スペクトル）
yf_norm_both_sides = np.abs(yf) / (N)  # 両側スペクトル正規化 (Nで除算)

# sample frequencies (n: window length, d: sample spacing)
# f = [0, 1, ...,   n/2-1,     -n/2, ..., -1] / (d*n)   if n is even
# f = [0, 1, ..., (n-1)/2, -(n-1)/2, ..., -1] / (d*n)   if n is odd
xf = fftfreq(n=N, d=d)

import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=xf, y=yf_norm_both_sides, name='周波数スペクトル (両側正規化)'))
fig.update_layout(showlegend=True, title='周波数スペクトル (scipy.fft)', xaxis_title='周波数 [Hz]', yaxis_title='振幅')
fig.show()

In [ ]:
# 振幅正規化（片側スペクトル）
yf_norm_one_side = np.abs(yf) / (N / 2)  # 片側スペクトル用に正規化 (N/2で除算)

# 片側周波数 (0 Hz以上)
xf_one_side = fftfreq(n=N, d=d)[:N//2]

import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=xf_one_side, y=yf_norm_one_side[:N//2], name='周波数スペクトル (片側正規化)'))
fig.update_layout(showlegend=True, title='周波数スペクトル (scipy.fft)', xaxis_title='周波数 [Hz]', yaxis_title='振幅')
fig.show()

In [ ]:
# 切り下げ除算の例
5//2

## 5.3 SciPyのInverse-FFT

scipy.fft.ifft<br>
https://docs.scipy.org/doc/scipy/reference/generated/scipy.fft.ifft.html#scipy.fft.ifft

確認結果：<br>
* 元の合成正弦波が再現されている。
* SciPyを使って、正しくInverse-FFT処理を実行できている。

In [ ]:
import scipy.fft
y = scipy.fft.ifft(yf)
print(f"{type(y)=}")
print(f"{type(y[0])=}")

import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=time_series, y=y.real, name='逆フーリエ変換結果 (real)'))
fig.add_trace(go.Scatter(x=time_series, y=y.imag, name='逆フーリエ変換結果 (imag)', mode='lines', line=dict(dash='dash')))
fig.update_layout(showlegend=True, title='逆フーリエ変換結果 (scipy.fft.ifft)', xaxis_title='時間 [s]', yaxis_title='振幅')
fig.show()

## 5.4 実数FFT (rfft / irfft)

「実数の時系列データ（物理量）」を扱う場合は、rfft（実数FFT）を使うのが一般的。

* fft: 負の周波数も含めた「左右対称」の結果を返す（データ数は $N$ 個）
* rfft: 正の周波数部分だけを効率よく返す（データ数は $N/2 + 1$ 個）
    * これに対応する周波数軸生成関数は rfftfreq 
    * 計算量もメモリも約半分で済むため、実務ではこちらが好まれる

scipy.fft.rfft<br>
https://docs.scipy.org/doc/scipy/reference/generated/scipy.fft.rfft.html#scipy.fft.rfft

scipy.fft.rfftfreq<br>
https://docs.scipy.org/doc/scipy/reference/generated/scipy.fft.rfftfreq.html

scipy.fft.irfft<br>
https://docs.scipy.org/doc/scipy/reference/generated/scipy.fft.irfft.html#scipy.fft.irfft

In [ ]:
from scipy.fft import rfft, rfftfreq
import numpy as np

# Number of sample points
N = time_series.shape[-1]
print(f"{N=}")

# sample spacing
d = dt  # [s]

# spectrum : complex ndarray (a: array_like)
yf_r = rfft(sin_wave4)  # rfftを使う
print(f"{type(yf_r)=}")
print(f"{type(yf_r[0])=}")
print(f"{yf_r.shape=}")

# 振幅正規化（片側スペクトル）
yf_r_norm = np.abs(yf_r) / (N / 2)  # 片側スペクトル正規化 (N /2 で除算)

# sample frequencies (n: window length, d: sample spacing)
# f = [0, 1, ...,     n/2-1,     n/2] / (d*n)   if n is even
# f = [0, 1, ..., (n-1)/2-1, (n-1)/2] / (d*n)   if n is odd
xf_r = rfftfreq(n=N, d=d)

import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=xf_r, y=yf_r_norm, name='周波数スペクトル (片側正規化)'))
fig.update_layout(showlegend=True, title='周波数スペクトル (scipy.rfft)', xaxis_title='周波数 [Hz]', yaxis_title='振幅')
fig.show()

In [ ]:
import scipy.fft
y_r = scipy.fft.irfft(yf_r)  # irfftを使う
print(f"{type(y_r)=}")
print(f"{type(y_r[0])=}")

import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=time_series, y=y_r.real, name='逆フーリエ変換結果 (real)'))
fig.add_trace(go.Scatter(x=time_series, y=y_r.imag, name='逆フーリエ変換結果 (imag)', mode='lines', line=dict(dash='dash')))
fig.update_layout(showlegend=True, title='逆フーリエ変換結果 (scipy.fft.irfft)', xaxis_title='時間 [s]', yaxis_title='振幅')
fig.show()

# 6. 用語

* 振幅スペクトル： 単位は [$\mathrm{V}$] など。信号の強さをそのまま表す。（上記で作成したもの）
* パワースペクトル： 単位は [$\mathrm{V}^2$] など。振幅の「2乗」をとったもの。エネルギーの強さを表す。